# CAMP — Curvature Adaptive Monge Patching  
### SurgBioMech · Version 0.1

This notebook provides a preliminary release of the `camp` pipeline so that users can generate results ahead of the full GitHub release.

**Workflow overview:**
1. **Section A** — Preprocess thoracic aorta meshes exported from ScanIP (`.mat` format): open terminal caps and remesh.
2. **Section B** — Run the `camp` algorithm on individual aortas or in a parameter sweep across cohorts.
3. **Section B.3** — Interactively visualize and compare `camp` output metrics across cohorts and parameter settings.

> **Prerequisites:** Ensure `thoracic_prepro.py` (and `cl.py` and `cutclean.py`) and `campsrc.py` are present in the same directory as this notebook before running any cells.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from thoracic_prepro import *

## A. Thoracic Aorta Preprocessing

Converts ScanIP-exported meshes (`.mat` format) into analysis-ready meshes by:
- Opening the ascending and descending **terminal caps** using a centerline + segmentation mask (`.mhd`) approach.
- **Remeshing** to a target face count.

**Output:** Processed meshes are saved to each scan's `\mesh` subfolder with a suffix indicating the target face count (e.g., `_o50K` for 50,000 elements).

**Key parameters:**
| Parameter | Description |
|---|---|
| `cohorts` | List of cohort codes to process (subdirectories under `base_root`) |
| `base_root` | Root directory containing all cohort data |
| `iso_spacing_mm` | Isotropic voxel spacing used when resampling the segmentation mask |
| `buffer_mm` | Half-width (mm) of the z-band used to identify and remove terminal cap patches |
| `target_faces` | Target number of faces after remeshing |
| `individ_scans` | Set to a list of specific scan IDs to process only those; `None` processes all |

> Skip this section if your meshes have already been preprocessed.

In [ ]:
spec_batch = None
log = thoracic_preprocessing_batch(
    cohorts=["JP"],
    base_root=r"Z:\aorta\thoracic",
    iso_spacing_mm=3.0,
    buffer_mm=5.0,
    target_faces=50_000,
    individ_scans=spec_batch,
    include_centerline_in_log=False,
    continue_on_error=True,
)
log

---
## B. Curvature Adaptive Monge Patching (`camp`)

`camp` is an adaptive patch-based surface analysis pipeline. It partitions a mesh surface into locally coherent geometric domains using intrinsic curvature properties.

**Algorithm summary:**
- Builds adaptive geodesic patches with curvature-weighted target sizes using local geodesic neighborhoods and **Monge 2nd-order polynomial fits**.
- Computes per-patch curvature metrics:
  - `avg_kg` — average discrete Gaussian curvature per patch
  - `integrated_K` — integrated Gaussian curvature per patch
- Iteration terminates when **every patch either passes both quality thresholds** (`R² ≥ R2_thresh` **and** `η ≤ eta_thresh`) **or is too small to split further** (`< 2 × min_patch_size` vertices). The final patch count is determined by the geometry, not a user-set target.

**Key quality parameters:**
| Parameter | Description |
|---|---|
| `R2_thresh` | Minimum R² for the Monge fit; higher = stricter planarity requirement |
| `eta_thresh` | Maximum shape index; lower = stricter curvature homogeneity |
| `min_patch_size` | Minimum number of vertices allowed in a patch |
| `seed_patches` | Initial number of seed patches |
| `trim_frac` | Optional fraction of extreme-curvature faces to trim before patching; `None` skips trimming |

In [2]:
import re
from campsrc import *

### B.1 Run `camp` on a Single Aorta

Set `scan_id` to the desired scan identifier. The code automatically parses the cohort letters and patient number from the ID and constructs the expected `.mat` file path.

The `out` dictionary returned by `camp()` contains the patch DataFrame, per-patch metrics, and an interactive Plotly figure. Call `out["figure"].show()` to render the colored surface.

In [3]:
scan_id = "KK1_1"
m = re.match(r"([A-Za-z]+)(\d+)(?:_(\d+))?$", scan_id)
letters, number = m.group(1), m.group(2)
patient_id = f"{letters}{number}"
MAT_PATH = fr"Z:\aorta\thoracic\{letters}\{patient_id}\{scan_id}\mesh\{scan_id}_o50K.mat"

out = camp(
    mat_path=MAT_PATH,
    scan_prefix=None,
    color_metric="integrated_K",
    color_bounds=(-0.5, 0.5),
    R2_thresh=0.9,
    eta_thresh=0.075,
    min_patch_size=18,
    seed_patches=100,
    trim_frac=None,
    color_scale="Inferno",
)
out["figure"].show()

[input] Loaded mesh: 25000 verts, 49803 faces
[converge] failed quality (R2+eta): 12/718 (1.7%)
[converge] at min size (unsplittable): 616/718 (85.8%)
[done] 718 patches after splits=92
[trim] trimmed patches: 0/718 (area frac=0.0000)
[kg] avg_kg      : [-0.005463, 0.008518]
[K] integrated_K: [-0.378, 0.4125]
[eta] range     : [0.0004845, 0.105]
[R2] range      : [0.6039, 0.9986]


### B.2 Parameter Sweep Across Cohorts

`camp_param_sweep` runs `camp` across all scans in the specified cohorts for every combination of `trim_fracs`. Results are returned as a single tidy DataFrame with one row per (scan, parameter-combination).

> **Note:** `n_jobs=6` parallelises across scans using joblib. Set to `-1` to use all available CPU cores, but be mindful of other running processes.

In [ ]:
csweep = camp_param_sweep(
    cohorts=("KK", "KY"),
    base_root=r"Z:\aorta\thoracic",
    mat_suffix="_o50",
    r2_thresh=0.9,
    eta_thresh=0.075,
    min_patch_size=18,
    seed_patches=100,
    trim_fracs=(0.0, 0.001, 0.005, 0.01, 0.05),
    n_jobs=6, #can set this to -1 to use all available CPU cores, but be careful if you have other processes running
)

### B.3 Interactive Visualization of `camp` Results

This section builds an interactive scatter plot widget for exploring `camp` output metrics across cohorts and parameter settings. 

**Before running the visualization cell**, load data into `df` using **one** of the two options below:
- **Option 1** — Use results from a sweep just completed in B.2 (`csweep`).
- **Option 2** — Load a previously saved CSV file.

The widget provides:
- **Dropdown selectors** for the x- and y-axis metrics.
- **Parameter sliders** to filter to a specific parameter combination.
- **Lock axes** checkbox to keep axis ranges fixed across slider changes (useful for comparing across parameter settings).
- **Straws only** checkbox to filter to straw-shaped patches (if `is_straw` column is present).

> To add a new cohort to the plot, find the `for i, c in enumerate(cohorts):` loop in the code cell below and add a new `if c == "XX":` block with the desired display name and color.

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

**Option 1 — Use results from the B.2 sweep (run B.2 first):**

In [ ]:
df = csweep.copy()

**Option 2 — Load a previously saved CSV file** (update the path as needed):

In [7]:
df = pd.read_csv(r"Z:\aorta\python_code\camp\UC_plus_acute_trials_camp71426.csv")

In [8]:
df["cohort"] = df["scan_id"].astype(str).str.extract(r"^([A-Za-z]+)")[0]
PARAM_COLS = ["r2_thresh", "eta_thresh", "min_patch_size", "seed_patches", "trim_frac"]
opts = {p: sorted(df[p].dropna().unique().tolist()) for p in PARAM_COLS}
feature_cols = [
    c for c in df.columns
    if c not in PARAM_COLS + ["scan_id", "cohort"] and pd.api.types.is_numeric_dtype(df[c])
]

def _default(col, fallback):
    return col if col in feature_cols else fallback

x_dd = widgets.Dropdown(options=feature_cols, value=_default("n_patches", feature_cols[0]), description="x:")
y_dd = widgets.Dropdown(options=feature_cols, value=_default("k_fluct", feature_cols[0]), description="y:")
lock = widgets.Checkbox(value=True, description="lock axes (global range)")
straws_only = widgets.Checkbox(value=("is_straw" in df.columns), description="straws only")

sliders = {
    p: widgets.SelectionSlider(options=opts[p], value=opts[p][len(opts[p]) // 2],
                               description=p, continuous_update=False,
                               style={"description_width": "120px"},
                               layout=widgets.Layout(width="480px"))
    for p in PARAM_COLS
}

cohorts = sorted(df["cohort"].dropna().unique().tolist())
fig = go.FigureWidget()

# Here is where you can add new cohorts to the plot. Just add a new if statement with the cohort code and the desired name and color.
for i, c in enumerate(cohorts):
    if c == "MF":
        fig.add_scatter(x=[], y=[], mode="markers", name="French MRA",
                        marker=dict(size=10, opacity=1, color="#8400ff"))
    if c == "KY":
        fig.add_scatter(x=[], y=[], mode="markers", name="Normal Controls",
                        marker=dict(size=5, opacity=0.5, color="#1F763A"))
    if c == "AV":
        fig.add_scatter(x=[], y=[], mode="markers", name="GORE 0802",
                        marker=dict(size=5, opacity=1, color="#49D397"))
    if c == "CT":
        fig.add_scatter(x=[], y=[], mode="markers", name="Cook",
                        marker=dict(size=5, opacity=1, color="#ff0000"))
    if c == "KK":
        fig.add_scatter(x=[], y=[], mode="markers", name="UC TBAD",
                        marker=dict(size=5, opacity=1, color="#ff7f0e"))
    if c == "DJ":
        fig.add_scatter(x=[], y=[], mode="markers", name="GORE 0801",
                        marker=dict(size=5, opacity=1, color="#0d00ff"))
    if c == "KP":
        fig.add_scatter(x=[], y=[], mode="markers", name="Medtronic",
                        marker=dict(size=5, opacity=1, color="#acb800"))

fig.update_layout(width=820, height=600, template="plotly_white",
                  legend=dict(title="cohort"), margin=dict(l=60, r=20, t=60, b=50))

def update(*_):
    sel = df
    for p in PARAM_COLS:
        sel = sel[sel[p] == sliders[p].value]
    if straws_only.value and "is_straw" in df.columns:
        sel = sel[sel["is_straw"].astype(bool)]
    xcol, ycol = x_dd.value, y_dd.value
    with fig.batch_update():
        for i, c in enumerate(cohorts):
            d = sel[sel["cohort"] == c]
            fig.data[i].x = d[xcol]
            fig.data[i].y = d[ycol]
            fig.data[i].text = d["scan_id"]
            fig.data[i].hovertemplate = f"%{{text}}<br>{xcol}=%{{x:.4g}}<br>{ycol}=%{{y:.4g}}<extra>{c}</extra>"
        fig.layout.xaxis.title = xcol
        fig.layout.yaxis.title = ycol
        if lock.value:
            fig.layout.xaxis.range = [df[xcol].min(), df[xcol].max()]
            fig.layout.yaxis.range = [df[ycol].min(), df[ycol].max()]
        else:
            fig.layout.xaxis.autorange = True
            fig.layout.yaxis.autorange = True
        params = "  ".join(f"{p}={sliders[p].value}" for p in PARAM_COLS)
        fig.layout.title = f"{params}   (n={len(sel)})"

for w in [x_dd, y_dd, lock, straws_only] + list(sliders.values()):
    w.observe(update, names="value")
update()

display(widgets.VBox([
    widgets.HBox([x_dd, y_dd]),
    widgets.HBox([lock, straws_only]),
    *[sliders[p] for p in PARAM_COLS],
    fig,
]))